# 🎙️ speakerscribe — Transcripción + diarización en Colab (producción)

**Flujo (v6, sin reinicio de runtime):** montar Drive → instalar desde el **wheel local** *(rápida; puede pedir **un** reinicio planificado)* → configurar → preflight → *(smoke)* → batch → resultados → **entregables** → *(renombrar · bench)* → liberar GPU → **apagar runtime**.

**Leyenda:** 🟢 celda **obligatoria** · 🟡 celda **opcional**

## 🚀 ¿Qué hace este pipeline?

| Capacidad | Detalle |
|---|---|
| ✂️ Inferencia *batched* | `batch_size` procesa varios segmentos a la vez en la T4 (2-4x vs secuencial) |
| 🧠 Diarización sobre el audio completo | mejor consistencia de hablantes en grabaciones largas |
| 🔁 Caché de diarización por parámetros | cambiar `num_speakers` invalida el caché automáticamente |
| 📊 Telemetría por etapa | cada `.json` trae tiempos de extracción, diarización y transcripción |
| 🚫 Filtro de muletillas | modos `safe`/`aggressive` sobre el `.transcript.md` |
| 📚 Glosario global y por archivo | `initial_prompt` (≤4000 chars) + `data/<nombre>.prompt.txt` |
| 📄 Salida unificada para LLMs | `<base>_full_for_llm.txt` con toda la transcripción |
| 💾 Idempotencia por contenido | historial SQLite + ledger: re-ejecutar solo procesa lo nuevo |
| 🛟 Degradación controlada | `ok_degraded` + auto-reintento de fallas CRÍTICAS en el siguiente batch |

## ✅ Antes de empezar — checklist

1. **Runtime con GPU**: `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.
2. **Token de Hugging Face** (tipo *Read*, no fine-grained) guardado en **Secrets** de Colab como `HF_TOKEN` (icono 🔑, panel izquierdo), con los términos aceptados en <https://huggingface.co/pyannote/speaker-diarization-community-1>.
3. **Audios/videos** dentro de `/content/drive/MyDrive/Pruebas/Speakerscribe/data/` — formatos: `.mp3`, `.mp4`, `.wav`, `.m4a`, `.ogg`, `.flac`, `.webm`.
4. **Reinicio planificado**: la CELDA 2 puede reiniciar el runtime **una vez** (lo instalado persiste). Cuando vuelva, re-ejecuta CELDAS 1–2 — la 2 termina en segundos — y sigue normal.
5. *(Opcional)* **Glosario por archivo**: `data/<nombre>.prompt.txt` junto al medio, con nombres propios y términos técnicos (los más importantes **al final**: Whisper solo condiciona con los ~224 tokens finales).

## 🟢 1 · Montar Google Drive

Va primero porque la librería (el `.whl`) vive en tu Drive: sin montarlo, la celda de instalación no puede encontrarla.

In [1]:
# 🟢 CELDA 1 — Drive (librería + outputs duraderos); los WAV temporales van a disco local
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 🟢 2 · Instalación rápida desde el wheel local (a lo sumo **un** reinicio planificado)

`RUTA_LIBRERIA` apunta a la carpeta de Drive con la librería; la celda localiza `dist/speakerscribe-*.whl` y lo instala **directo, sin constraints** (~2-3 min) — la vía que ya demostró funcionar. No agregues `!pip install` manuales: el wheel arrastra `loguru`, `faster-whisper`, `pyannote.audio`, etc.

**Por qué puede pedir un reinicio (y por qué está bien):** `pyannote.audio` 4.0.x ancla `torch` a una versión **exacta** (p. ej. `torch==2.8.0`), distinta a la que trae Colab — congelar versiones es matemáticamente irresoluble, así que pip elige un pyannote compatible con tu torch y a veces **actualiza numpy**, un binario que la sesión ya tiene cargado. La celda lo detecta y reinicia el runtime **una sola vez** (todo lo instalado persiste). Al volver, re-ejecuta CELDAS 1–2: la 2 entra por su *fast-path* y termina en segundos con verificación completa (`pip check` filtrado al stack + origen del código + versión).

> ⚠️ Sigue vigente: no hagas `%cd` a la carpeta de la librería — el código fuente de desarrollo en Drive ensombrece al paquete instalado (así ocurrió el error `apply_seed`, con el batch 0/6). La celda sanea `sys.path`/CWD sola y verifica el origen del import.

**Por qué desde el wheel y no `pip install speakerscribe`:** PyPI solo publica la **0.1.1**; tu copia es la **0.3.0** y este notebook usa su API.

In [ ]:
# 🟢 CELDA 2 — instalación rápida del wheel local (a lo sumo UN reinicio planificado)
REINICIO_AUTOMATICO = True  # ⇦ False = la celda solo avisa y tú reinicias manualmente

import importlib, importlib.metadata as im
import os, subprocess, sys, time
from pathlib import Path

RUTA_LIBRERIA = '/content/drive/MyDrive/Pruebas/Speakerscribe'  # ⇦ carpeta de Drive con la librería (dist/*.whl)
lib = Path(RUTA_LIBRERIA).resolve()

# ── helpers ─────────────────────────────────────────────────────────
def _dentro_de_lib(p: str) -> bool:
    try:
        rp = Path(p or '.').resolve()
    except OSError:
        return False
    return rp == lib or lib in rp.parents

def _ver(pkg: str) -> str | None:
    try:
        return im.version(pkg)
    except im.PackageNotFoundError:
        return None

def _pip_check_stack() -> None:
    """pip check filtrado a nuestro stack (Colab base trae quejas ajenas)."""
    r = subprocess.run([sys.executable, '-m', 'pip', 'check'],
                       capture_output=True, text=True)
    stack = ('speakerscribe', 'faster-whisper', 'ctranslate2', 'pyannote',
             'torch', 'numpy', 'pydantic', 'loguru', 'typer', 'rich',
             'lightning', 'torchmetrics')
    lineas = [l for l in r.stdout.splitlines() if any(s in l.lower() for s in stack)]
    if lineas:
        print('⚠️  pip check reporta sobre nuestro stack (mira la CELDA 2b):')
        for l in lineas:
            print('   ', l)
    else:
        print('✔ pip check: stack coherente (sin conflictos declarados).')

def _verificar_import() -> str:
    import speakerscribe
    origen = Path(speakerscribe.__file__).resolve()
    if origen == lib or lib in origen.parents:
        raise RuntimeError(f'El import resolvió a la copia de Drive ({origen}), no a la instalada. '
                           'Reinicia el runtime y ejecuta desde la CELDA 1.')
    assert speakerscribe.__version__ == wheel_ver, (
        f'import resolvió v{speakerscribe.__version__}, se esperaba v{wheel_ver}. '
        'Reinicia el runtime y ejecuta desde la CELDA 1.')
    return speakerscribe.__version__

# 1) Higiene de imports: la carpeta de la librería no puede ser el CWD ni estar
#    en sys.path — una carpeta speakerscribe/ ahí ensombrece al paquete instalado
#    (así corrió el código de desarrollo que causó el error apply_seed).
sys.path[:] = [p for p in sys.path if not (p and _dentro_de_lib(p))]
if _dentro_de_lib(os.getcwd()):
    os.chdir('/content')
    print('CWD movido a /content (estaba dentro de la carpeta de la librería).')

# 2) Localizar el wheel en dist/
wheels = list(lib.rglob('speakerscribe-*.whl'))
if not wheels:
    raise FileNotFoundError(
        f'No hay ningún speakerscribe-*.whl bajo {RUTA_LIBRERIA} — '
        'verifica que la carpeta dist/ exista con el wheel adentro.')
wheel = max(wheels, key=lambda p: tuple(int(x) for x in p.name.split('-')[1].split('.') if x.isdigit()))
wheel_ver = wheel.name.split('-')[1]

CRITICOS = ('numpy', 'torch', 'torchaudio', 'torchvision')
t0 = time.time()
instalada = _ver('speakerscribe')

if instalada == wheel_ver:
    # ⚡ Fast-path (estado normal tras el reinicio planificado): verificar y seguir.
    v = _verificar_import()
    _pip_check_stack()
    print(f'⚡ speakerscribe v{v} ya instalado y verificado en {time.time() - t0:.1f} s — sigue con la CELDA 3.')
else:
    if instalada:
        print(f'Instalada v{instalada} ≠ wheel v{wheel_ver} → se reemplaza por la del wheel.')
    antes = {p: _ver(p) for p in CRITICOS}
    ya_importado = 'speakerscribe' in sys.modules
    print(f'Instalando {wheel.name} + dependencias (resolución libre, ~2-3 min)...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', str(wheel)],
                       capture_output=True, text=True)
    if r.returncode != 0:  # pip NUNCA debe fallar en silencio
        print('──── salida de pip (final) ────')
        print(r.stdout[-6000:]); print(r.stderr[-4000:])
        raise RuntimeError('pip falló instalando el wheel — el detalle está arriba; '
                           'ejecuta la CELDA 2b y comparte ambas salidas.')
    importlib.invalidate_caches()
    despues = {p: _ver(p) for p in CRITICOS}
    deriva = [f'{p}: {antes[p]} → {despues[p]}'
              for p in CRITICOS if antes[p] and despues[p] and antes[p] != despues[p]]
    if ya_importado and instalada and instalada != wheel_ver:
        deriva.append(f'speakerscribe: el módulo ya estaba importado con v{instalada}')

    if deriva:
        print(f'Instalación OK en {time.time() - t0:.0f} s. '
              'pip actualizó binarios que la sesión ya tenía cargados:')
        for d in deriva:
            print('   ♻️ ', d)
        print()
        print('══════════════════════════════════════════════════════════════')
        print('  REINICIO PLANIFICADO (1 de 1) — todo lo instalado PERSISTE.')
        print('  Cuando el runtime vuelva (~10 s), ejecuta de nuevo la CELDA 1')
        print('  y la CELDA 2 (esta vez termina en segundos) y continúa normal.')
        print('══════════════════════════════════════════════════════════════')
        if REINICIO_AUTOMATICO:
            sys.stdout.flush()
            time.sleep(5)              # da tiempo a que el mensaje se muestre
            os.kill(os.getpid(), 9)    # Colab reinicia el kernel solo; paquetes intactos
        raise RuntimeError('Reinicia el runtime (Entorno de ejecución → Reiniciar sesión) '
                           'y re-ejecuta las CELDAS 1–2.')

    v = _verificar_import()
    _pip_check_stack()
    print(f'speakerscribe v{v} listo desde {wheel.name} en {time.time() - t0:.0f} s (sin reinicio necesario).')

Instalando speakerscribe-0.3.0-py3-none-any.whl + dependencias (resolución libre, ~2-3 min)...
Instalación OK en 33 s. pip actualizó binarios que la sesión ya tenía cargados:
   ♻️  numpy: 2.0.2 → 2.4.6

══════════════════════════════════════════════════════════════
  REINICIO PLANIFICADO (1 de 1) — todo lo instalado PERSISTE.
  Cuando el runtime vuelva (~10 s), ejecuta de nuevo la CELDA 1
  y la CELDA 2 (esta vez termina en segundos) y continúa normal.
══════════════════════════════════════════════════════════════


### 🟡 2b · Diagnóstico del entorno (ejecutar solo si algo falla)

In [2]:
# 🟡 CELDA 2b — diagnóstico: versiones críticas e incompatibilidades
import importlib, importlib.metadata as im, shutil, sys

print(f'Python {sys.version.split()[0]}')
critical = {}
for pkg in ('speakerscribe', 'faster-whisper', 'ctranslate2', 'pyannote.audio',
            'torch', 'numpy', 'pydantic', 'loguru'):
    try:
        critical[pkg] = im.version(pkg)
    except im.PackageNotFoundError:
        critical[pkg] = 'NO INSTALADO'
for k, v in critical.items():
    print(f'  {k:<18} {v}')

try:
    import torch
    print(f'  CUDA disponible    {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'  GPU                {torch.cuda.get_device_name(0)}')
except Exception as e:
    print(f'  torch import FALLÓ: {e}')
print(f"  ffmpeg             {'OK' if shutil.which('ffmpeg') else 'NO ENCONTRADO'}")

# El código debe venir de site-packages, NUNCA de la copia en Drive
try:
    import speakerscribe
    origen = speakerscribe.__file__
    print(f"  origen del código  {'⚠️ DRIVE (mal)' if '/content/drive' in origen else 'site-packages (ok)'}")
    print(f'                     {origen}')
except Exception as e:
    print(f'  import speakerscribe FALLÓ: {e}')

# pip check filtrado a nuestro stack (la base de Colab trae quejas ajenas)
import subprocess
_r = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
_stack = ('speakerscribe', 'faster-whisper', 'ctranslate2', 'pyannote', 'torch',
          'numpy', 'pydantic', 'loguru', 'typer', 'rich', 'lightning', 'torchmetrics')
_quejas = [l for l in _r.stdout.splitlines() if any(s in l.lower() for s in _stack)]
print('\npip check (stack):', 'sin conflictos declarados ✔' if not _quejas else '')
for l in _quejas:
    print('   ⚠️', l)

problems = [k for k, v in critical.items() if v == 'NO INSTALADO']
if problems:
    print(f'\nCRÍTICO: faltan {problems}.')
    print('Reinicia el runtime (Entorno de ejecución → Reiniciar) y vuelve a la CELDA 1.')
else:
    print('\nEntorno consistente: no se requiere reinicio.')

Python 3.12.13
  speakerscribe      0.3.0
  faster-whisper     1.2.1
  ctranslate2        4.8.0
  pyannote.audio     4.0.4
  torch              2.11.0+cu128
  numpy              2.4.6
  pydantic           2.12.3
  loguru             0.7.3
  CUDA disponible    True
  GPU                Tesla T4
  ffmpeg             OK
  origen del código  site-packages (ok)
                     /usr/local/lib/python3.12/dist-packages/speakerscribe/__init__.py

pip check (stack): 
   ⚠️ numba 0.60.0 has requirement numpy<2.1,>=1.22, but you have numpy 2.4.6.

Entorno consistente: no se requiere reinicio.


## 🟢 3 · Configuración — ✏️ EDITA AQUÍ (la única celda que requiere cambios)

Estructura esperada: `WORKSPACE/data/` con tus audios/videos. Todas las opciones están documentadas en la propia celda, agrupadas: modelo (con VRAM), diarización, glosario, cortes/limpieza, control de ejecución y **selección de entregables** (lo que la CELDA 8 copiará a `entregables/`). Pydantic valida cada valor al construir el config — un error de tipeo falla aquí, no a mitad del batch.

In [3]:
# 🟢 CELDA 3 — ✏️ EDITA AQUÍ: la única celda que requiere cambios
from pathlib import Path

from speakerscribe import TranscriptionConfig, WorkspacePaths
from speakerscribe.estimates import estimate_processing_minutes

# ── Workspace (carpeta raíz en Drive; debe contener data/ con tus audios) ──
WORKSPACE = '/content/drive/MyDrive/Pruebas/Speakerscribe'

# ── Modelo Whisper ──────────────────────────────────────────────────
#   'tiny'            ~1.5 GB VRAM — solo smoke tests
#   'base'            ~1.7 GB VRAM — pruebas rápidas
#   'small'           ~2.5 GB VRAM — rápido, calidad decente
#   'medium'          ~4.0 GB VRAM — intermedio
#   'large-v3'        ~5.0 GB VRAM — MÁXIMA calidad, más lento ⭐ CONFIGURADO
#   'large-v3-turbo'  ~3.5 GB VRAM — rápido + excelente precisión
config = TranscriptionConfig(
    # — Núcleo —
    model='large-v3',
    language='es',                 # None = autodetección ('es', 'en', 'pt', 'fr', ...)
    batch_size=8,                  # inferencia batched en T4 (1 = secuencial; baja a 4 si hay OOM)
    beam_size=5,                   # 1 = más rápido | 5 = más preciso (default)

    # — Diarización (identificación de hablantes) —
    enable_diarization=True,
    speaker_assignment='word',     # 'word' re-segmenta en cada cambio de hablante (recomendado)
    num_speakers=None,             # si CONOCES el número exacto → p. ej. 2 (mejora la precisión)
    min_speakers=2,                # ┐ solo se usan cuando num_speakers=None
    max_speakers=12,                # ┘ (reunión típica: 6 · entrevista 1-a-1: num_speakers=2)

    # — Glosario global (gran mejora en nombres propios; lo MÁS importante AL FINAL) —
    initial_prompt=None,           # p. ej. 'Snowflake, Bancóldex, GIC, ProColombia' (≤4000 chars)
                                   #   por archivo: data/<nombre>.prompt.txt (tiene prioridad)

    # — Cortes y limpieza del .transcript.md —
    vad_min_silence_ms=1000,       # silencios ≥ N ms cortan segmento (default librería: 500)
    gap_max_s_transcript=3.0,      # pausa > N s del mismo hablante abre bloque nuevo (default: 2.0)
    remove_fillers='safe',         # 'off' | 'safe' | 'aggressive' (quita sí/no/claro — ¡cuidado!)

    # — Control de ejecución —
    force_reprocess=False,         # True = ignora outputs existentes y rehace todo
    auto_retry_on_critical=True,   # reintenta en el siguiente batch las corridas con flag CRÍTICO
    evaluate_quality=True,         # chequeos heurísticos de calidad post-corrida
    enable_runs_db=True,           # historial SQLite → idempotencia por hash de contenido
    produce_unified_for_llm=True,  # genera <base>_full_for_llm.txt (transcripción completa)
)

# ── Entregables: qué copiar a entregables/ en la CELDA 8 ───────────
COPY_TRANSCRIPT_MD = True    # ⭐ Markdown legible para humanos
COPY_FULL_FOR_LLM  = True    # ⭐ transcripción completa para LLMs de contexto largo
COPY_SPLITS        = False   # solo si tu LLM tiene ventana de contexto corta
COPY_JSON          = False   # solo si necesitas timestamps/metadata detallada
COPY_SRT           = False   # solo si necesitas subtítulos para video

# ── Preparar workspace y listar data/ ───────────────────────────────
paths = WorkspacePaths(workspace=WORKSPACE)  # scratch local automático
paths.create_directories()
media = paths.list_media_files()
print(f'{len(media)} archivo(s) en data/:')
for f in media:
    print(f'   {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')
if not media:
    print(f'⚠️  data/ está vacío — sube tus audios/videos a: {paths.data}')

# ── Resumen de ajustes (verifícalos antes de seguir) ────────────────
print('\n📂 Workspace :', WORKSPACE)
print('🧠 Modelo    :', config.model, '| idioma:', config.language or 'auto',
      '| batch:', config.batch_size, '| beam:', config.beam_size)
print('👥 Hablantes :', config.num_speakers or f'{config.min_speakers}-{config.max_speakers}',
      '| asignación:', config.speaker_assignment)
glos = config.initial_prompt or '(vacío — usa data/<nombre>.prompt.txt por archivo)'
print('📚 Glosario  :', glos[:70] + ('…' if len(glos) > 70 else ''))
print('🚫 Muletillas:', config.remove_fillers, '| 📦 entregables: '
      f'md={COPY_TRANSCRIPT_MD}, full_llm={COPY_FULL_FOR_LLM}, splits={COPY_SPLITS}, '
      f'json={COPY_JSON}, srt={COPY_SRT}')
if media:
    est = estimate_processing_minutes(
        sum(f.stat().st_size for f in media) / 1e6 / 1.0,  # ~1 MB ≈ 1 min de voz
        config.model, with_diarization=config.enable_diarization)
    print(f'\n⏱️  Estimación (T4, planeación secuencial): ~{est:.0f} min; batched suele ser 2-4x más rápido.')
    print('   Tu RTF real queda en el ledger — confía en tu historial sobre esta tabla.')

1 archivo(s) en data/:
   2026-06-12 *Célula IA.mkv  (90.7 MB)

📂 Workspace : /content/drive/MyDrive/Pruebas/Speakerscribe
🧠 Modelo    : large-v3 | idioma: es | batch: 8 | beam: 5
👥 Hablantes : 2-12 | asignación: word
📚 Glosario  : (vacío — usa data/<nombre>.prompt.txt por archivo)
🚫 Muletillas: safe | 📦 entregables: md=True, full_llm=True, splits=False, json=False, srt=False

⏱️  Estimación (T4, planeación secuencial): ~30 min; batched suele ser 2-4x más rápido.
   Tu RTF real queda en el ledger — confía en tu historial sobre esta tabla.


## 🟢 4 · Pre-flight (valida TODO antes de cargar modelos)

In [4]:
# 🟢 CELDA 4 — preflight: archivos, disco (workspace y scratch), GPU/VRAM, token HF
from speakerscribe import preflight_check

info = preflight_check(paths, config)
info

2026-06-13 00:46:14.044 | INFO     | speakerscribe.pipeline:preflight_check:131 - Pre-flight check...
2026-06-13 00:46:14.047 | INFO     | speakerscribe.pipeline:preflight_check:142 -    1 file(s) — 90.7 MB total
2026-06-13 00:46:14.048 | INFO     | speakerscribe.pipeline:preflight_check:151 -    Disk (workspace): 66,276 MB free (~500 MB required)
2026-06-13 00:46:14.049 | INFO     | speakerscribe.pipeline:preflight_check:158 -    Disk (scratch /content/ss_scratch): 69,764 MB free
2026-06-13 00:46:14.050 | INFO     | speakerscribe.pipeline:preflight_check:185 -    VRAM: 15.6 GB free / 15.6 GB total
2026-06-13 00:46:14.051 | INFO     | speakerscribe.pipeline:preflight_check:189 -    Device: cuda (float16)
2026-06-13 00:46:17.357 | INFO     | speakerscribe.pipeline:preflight_check:206 -    HF token: detected


{'n_files': 1,
 'total_mb': 90.7,
 'free_mb': 66276.0,
 'scratch_free_mb': 69764.0,
 'gpu_ok': True,
 'vram_available_gb': 15.64,
 'hf_token_ok': True,
 'device': 'cuda',
 'compute_type': 'float16'}

## 🟡 5 · Smoke test (2-3 min, aislado en `_smoke/`) — *muy recomendada tras cambiar entorno o librería*

Corre el primer archivo con el modelo `small` hacia `WORKSPACE/_smoke/` — verifica el flujo completo **sin** tocar tus salidas reales. Un smoke habría detectado el error `apply_seed` en segundos, antes de lanzar el batch.

In [ ]:
# 🟡 CELDA 5 — smoke aislado en _smoke/ (modelo small, ~2-3 min)
import time

from speakerscribe import process_one, loaded_whisper

if not media:
    raise FileNotFoundError(f'No hay archivos en {paths.data} — sube al menos uno y re-ejecuta la CELDA 3.')

smoke_cfg = config.model_copy(update={
    'model': 'small', 'beam_size': 1, 'force_reprocess': True,
    'enable_runs_db': False, 'evaluate_quality': False,
})
smoke_paths = WorkspacePaths(workspace=str(Path(WORKSPACE) / '_smoke'))
smoke_paths.create_directories()

print(f'🧪 Smoke sobre: {media[0].name}')
t0 = time.time()
try:
    with loaded_whisper(smoke_cfg) as smoke_model:
        smoke = process_one(media[0], smoke_paths, smoke_model, smoke_cfg)
except Exception as e:
    print(f'❌ Smoke FALLÓ: {type(e).__name__}: {e}')
    raise
print(f'✅ Smoke OK en {time.time() - t0:.1f} s')
print(f"   Estado   : {smoke.get('status')}")
print(f"   Palabras : {smoke.get('total_words', 0):,} | idioma: {smoke.get('language_detected', '-')}")
print(f"   Tiempos  : {smoke.get('timings', {})}")
print('\n— Muestra del texto —')
print((smoke_paths.transcripts / f"{smoke['base_name']}.txt").read_text()[:600])

## 🟢 6 · Procesamiento del batch completo

Modelos cargados **una sola vez** para todo el lote (Whisper + pyannote residentes). Idempotente por contenido: re-ejecutar solo procesa lo nuevo. Si la diarización falla en un archivo, termina como `ok_degraded` (flag CRÍTICO) y se reintenta en el siguiente batch.

In [5]:
# 🟢 CELDA 6 — batch (modelos cargados UNA vez para todo el lote)
from speakerscribe import process_batch

results = process_batch(paths, config)

print(f'\n📁 Transcripciones : {paths.transcripts}')
print(f'📁 Splits          : {paths.splits}')
print(f'💾 Historial       : {paths.db_path.name} + {paths.ledger_path.name} (idempotencia por hash)')

2026-06-13 00:46:28.601 | INFO     | speakerscribe.pipeline:preflight_check:131 - Pre-flight check...
2026-06-13 00:46:28.605 | INFO     | speakerscribe.pipeline:preflight_check:142 -    1 file(s) — 90.7 MB total
2026-06-13 00:46:28.607 | INFO     | speakerscribe.pipeline:preflight_check:151 -    Disk (workspace): 66,276 MB free (~500 MB required)
2026-06-13 00:46:28.607 | INFO     | speakerscribe.pipeline:preflight_check:158 -    Disk (scratch /content/ss_scratch): 69,764 MB free
2026-06-13 00:46:28.609 | INFO     | speakerscribe.pipeline:preflight_check:185 -    VRAM: 15.6 GB free / 15.6 GB total
2026-06-13 00:46:28.609 | INFO     | speakerscribe.pipeline:preflight_check:189 -    Device: cuda (float16)
2026-06-13 00:46:29.039 | INFO     | speakerscribe.pipeline:preflight_check:206 -    HF token: detected
2026-06-13 00:46:29.041 | INFO     | speakerscribe.pipeline:process_batch:757 - 1 file(s) detected:
2026-06-13 00:46:29.043 | INFO     | speakerscribe.pipeline:process_batch:759 -   

config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

embedding/pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

2026-06-13 00:47:30.186 | DEBUG    | speakerscribe.diarization:_ensure_loaded:126 - VRAM after loading pyannote: 0.03 GB
2026-06-13 00:47:30.187 | SUCCESS  | speakerscribe.diarization:_ensure_loaded:127 - pyannote pipeline loaded in 4.0s (stays resident)
2026-06-13 00:47:30.188 | INFO     | speakerscribe.diarization:diarize:190 -    Speakers: min=2 max=12
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction


📁 Transcripciones : /content/drive/MyDrive/Pruebas/Speakerscribe/transcripts
📁 Splits          : /content/drive/MyDrive/Pruebas/Speakerscribe/splits
💾 Historial       : _runs.db + _runs.jsonl (idempotencia por hash)


## 🟢 7 · Resultados e historial

In [6]:
# 🟢 CELDA 7 — resumen + historial del ledger
from speakerscribe.persistence import global_stats, list_runs

for r in results:
    print(f"{r.get('status','?'):<12} {r.get('audio_file','?'):<45} "
          f"RTF={r.get('real_time_factor','-')}")
print('\nGlobal:', global_stats(paths.ledger_path))
for run in list_runs(paths.ledger_path, limit=5):
    print(f"  {run.get('processed_at','')[:19]}  {run.get('audio_file',''):<40} "
          f"{run.get('status','')}")

ok           2026-06-12 *Célula IA_b47cea3ced.wav         RTF=24.67

Global: {'total_runs': 110, 'ok_runs': 104, 'ok_degraded_runs': 0, 'error_runs': 6, 'skipped_runs': 0, 'avg_rtf': 11.73, 'hours_processed': 78.65, 'total_words': 627005}
  2026-06-13T00:55:19  2026-06-12 *Célula IA_b47cea3ced.wav    ok
  2026-06-12T05:35:31  2026-06-11 Reunión Gestión Activos GIC_cd4eded145.wav ok
  2026-06-12T05:33:07  2026-06-11 *Revisión automatización planeación_ba2706ef1b.wav ok
  2026-06-12T05:26:57  2026-06-11 *Comité Analítica_c39c2a00e7.wav ok
  2026-06-12T05:18:33  2026-06-04 16-02-09 - Data Science training Snowflake_09c4257121.wav ok


### 🟡 7b · Inspeccionar la metadata de un archivo (opcional)

Vistazo rápido al `.json` de una corrida: duración, idioma, hablantes, palabras, RTF, flags de calidad y tiempos por etapa.

In [7]:
# 🟡 CELDA 7b — vistazo a la metadata de una corrida
from speakerscribe.maintenance import inspect_json

ARCHIVO_JSON = None  # ⇦ ruta a un .json concreto, o None = usar el más reciente
jsons = sorted(paths.transcripts.glob('*.json'), key=lambda p: p.stat().st_mtime)
objetivo = Path(ARCHIVO_JSON) if ARCHIVO_JSON else (jsons[-1] if jsons else None)
if objetivo:
    print(f'📋 Inspeccionando: {objetivo.name}\n')
    info = inspect_json(objetivo)
else:
    print('⚠️  Aún no hay .json en transcripts/ — ejecuta antes la CELDA 6.')

2026-06-13 00:55:20.186 | INFO     | speakerscribe.maintenance:inspect_json:123 - 2026-06-12 *Célula IA_large-v3.json
2026-06-13 00:55:20.187 | INFO     | speakerscribe.maintenance:inspect_json:125 -    audio_file              : 2026-06-12 *Célula IA_b47cea3ced.wav
2026-06-13 00:55:20.188 | INFO     | speakerscribe.maintenance:inspect_json:125 -    model                   : large-v3
2026-06-13 00:55:20.189 | INFO     | speakerscribe.maintenance:inspect_json:125 -    diarization_enabled     : True
2026-06-13 00:55:20.191 | INFO     | speakerscribe.maintenance:inspect_json:125 -    speakers_summary        : {'SPEAKER_01': 92, 'SPEAKER_02': 28, 'SPEAKER_00': 18, 'SPEAKER_03': 75, 'SPEAKER_NO_OVERLAP': 11, 'SPEAKER_04': 26}
2026-06-13 00:55:20.191 | INFO     | speakerscribe.maintenance:inspect_json:125 -    total_segments          : 250
2026-06-13 00:55:20.193 | INFO     | speakerscribe.maintenance:inspect_json:125 -    total_words             : 6140
2026-06-13 00:55:20.193 | INFO     | 

📋 Inspeccionando: 2026-06-12 *Célula IA_large-v3.json



## 🟢 8 · Entregables + resumen consolidado

Copia a `entregables/` **solo** los archivos finales que marcaste en la CELDA 3 (tu carpeta limpia de consumo) y genera dos salidas de auditoría:

- **`_resumen.md`**: tabla legible con todos los archivos procesados (duración, idioma, hablantes, palabras, RTF, flags de calidad, tiempos por etapa).
- **`_metadata/run_<timestamp>.json`**: metadata completa de la corrida para trazabilidad y reproducibilidad (el token HF queda **redactado**).

> ⚠️ **No borres** `transcripts/` ni `splits/`: son la fuente de verdad de la idempotencia — borrarlas fuerza reprocesar todo.

In [8]:
# 🟢 CELDA 8 — entregables + _resumen.md + metadata de la corrida
import json as _json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import speakerscribe

if 'results' not in globals():
    raise RuntimeError('No hay resultados en memoria — ejecuta antes la CELDA 6 (batch).')

entregables = Path(WORKSPACE) / 'entregables'
entregables.mkdir(parents=True, exist_ok=True)
meta_dir = entregables / '_metadata'
meta_dir.mkdir(exist_ok=True)

OK_STATUSES = ('ok', 'ok_degraded')
n_ok   = sum(1 for r in results if r.get('status') == 'ok')
n_deg  = sum(1 for r in results if r.get('status') == 'ok_degraded')
n_skip = sum(1 for r in results if r.get('status') == 'skipped')
n_err  = sum(1 for r in results if r.get('status') == 'error')
n_words = sum(r.get('total_words') or 0 for r in results if r.get('status') in OK_STATUSES)

copied = []
for r in results:
    if r.get('status') not in OK_STATUSES:
        continue
    base = r.get('base_name')
    if not base:
        continue
    candidatos = []
    if COPY_TRANSCRIPT_MD:
        candidatos.append(paths.transcripts / f'{base}.transcript.md')
    if COPY_JSON:
        candidatos.append(paths.transcripts / f'{base}.json')
    if COPY_SRT:
        candidatos.append(paths.transcripts / f'{base}.srt')
    if COPY_FULL_FOR_LLM:
        candidatos.append(paths.splits / f'{base}_full_for_llm.txt')
    if COPY_SPLITS:
        candidatos.extend(s for s in sorted(paths.splits.glob(f'{base}_*.txt'))
                          if not s.name.endswith('_full_for_llm.txt'))
    for src in candidatos:
        if src.exists():
            shutil.copy2(src, entregables / src.name)
            copied.append(src.name)

# ── _resumen.md (tabla legible del batch) ─────────────────────────
ts_utc = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
lines = [
    f'# Resumen del batch — {ts_utc}',
    '',
    f'- **speakerscribe:** v{speakerscribe.__version__}',
    f'- **Modelo Whisper:** `{config.model}`',
    f'- **Idioma:** {config.language or "autodetección"}',
    f'- **Diarización:** {"sí" if config.enable_diarization else "no"}',
    '',
    '## Resultados por archivo',
    '',
    '| # | Audio | Estado | Duración | Idioma (conf) | Hablantes | Palabras | RTF | Flags de calidad |',
    '|---|---|---|---|---|---|---|---|---|',
]
for i, r in enumerate(results, 1):
    status, audio = r.get('status', '?'), r.get('audio_file', '?')
    if status in OK_STATUSES:
        icono = '✅' if status == 'ok' else '⚠️ degradado'
        flags = '; '.join(r.get('quality_flags') or []) or '—'
        if len(flags) > 60:
            flags = flags[:57] + '...'
        lines.append(
            f"| {i} | `{audio}` | {icono} | {r.get('duration_minutes') or 0:.1f} min "
            f"| {r.get('language_detected', '?')} ({r.get('language_probability') or 0:.0%}) "
            f"| {len(r.get('speakers_summary') or {})} | {r.get('total_words') or 0:,} "
            f"| {r.get('real_time_factor') or 0:.1f}x | {flags} |")
    elif status == 'skipped':
        lines.append(f'| {i} | `{audio}` | ⏩ omitido | — | — | — | — | — | (ya procesado) |')
    else:
        err = str(r.get('error', 'desconocido'))[:80]
        lines.append(f'| {i} | `{audio}` | ❌ error | — | — | — | — | — | {err} |')

lines += ['', '## Tiempos por etapa (segundos)', '',
          '| # | Audio | extract_wav | diarización | transcripción | transcript_md | total |',
          '|---|---|---|---|---|---|---|']
for i, r in enumerate(results, 1):
    if r.get('status') not in OK_STATUSES:
        continue
    t = r.get('timings') or {}
    lines.append(f"| {i} | `{r.get('audio_file', '?')}` | {t.get('extract_wav_s', '—')} "
                 f"| {t.get('diarization_s', '—')} | {t.get('transcribe_total_s', '—')} "
                 f"| {t.get('transcript_md_s', '—')} | {t.get('total_s', '—')} |")

resumen_path = entregables / '_resumen.md'
resumen_path.write_text('\n'.join(lines), encoding='utf-8')

# ── Metadata JSON de la corrida (token HF redactado) ──────────────
cfg_dump = config.model_dump()
if cfg_dump.get('hf_token'):
    cfg_dump['hf_token'] = '***redactado***'
run_metadata = {
    'timestamp_utc': ts_utc,
    'speakerscribe_version': speakerscribe.__version__,
    'config': cfg_dump,
    'results_summary': {'ok': n_ok, 'ok_degraded': n_deg, 'skipped': n_skip,
                        'errors': n_err, 'total_words': n_words},
    'results': [{k: r.get(k) for k in (
        'audio_file', 'status', 'base_name', 'duration_minutes', 'language_detected',
        'total_words', 'real_time_factor', 'chunked', 'n_chunks', 'speakers_summary',
        'quality_ok', 'quality_flags', 'timings', 'error')} for r in results],
}
meta_path = meta_dir / f"run_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.json"
meta_path.write_text(_json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'📦 Entregables en: {entregables}')
print(f'   • Archivos copiados : {len(copied)}')
for c in copied[:10]:
    print(f'     - {c}')
if len(copied) > 10:
    print(f'     ... y {len(copied) - 10} más')
print(f'   • Resumen del batch : {resumen_path.name}')
print(f'   • Metadata          : {meta_path.name}')
print(f'   • OK={n_ok}  degradados={n_deg}  omitidos={n_skip}  errores={n_err}  palabras={n_words:,}')
print('\n💡 Los originales quedan intactos en transcripts/ y splits/ — no los borres (rompe la idempotencia).')

📦 Entregables en: /content/drive/MyDrive/Pruebas/Speakerscribe/entregables
   • Archivos copiados : 2
     - 2026-06-12 *Célula IA_large-v3.transcript.md
     - 2026-06-12 *Célula IA_large-v3_full_for_llm.txt
   • Resumen del batch : _resumen.md
   • Metadata          : run_20260613_005520.json
   • OK=1  degradados=0  omitidos=0  errores=0  palabras=6,140

💡 Los originales quedan intactos en transcripts/ y splits/ — no los borres (rompe la idempotencia).


## 🟡 9 · Renombrar hablantes (opcional)

Escucha el inicio del audio, identifica quién es `SPEAKER_00`, etc., edita `MAPPING` y pon `RUN_RENAME = True`. El cambio se aplica a TODOS los formatos del `BASE_NAME` (txt, srt, md, json, splits) — los intercambios son seguros (una sola pasada) — y las copias de `entregables/` de ese archivo se refrescan solas.

In [ ]:
# 🟡 CELDA 9 — renombrar hablantes (edita MAPPING y pon RUN_RENAME=True)
import shutil

from speakerscribe.maintenance import rename_speakers_in_outputs

disponibles = sorted({f.stem.replace('.transcript', '')
                      for f in paths.transcripts.glob('*.transcript.md')})
print('📋 Base names disponibles:')
for b in disponibles:
    print(f'   • {b}')

# ✏️ Edita estos valores:
BASE_NAME = disponibles[-1] if disponibles else 'tu_audio_modelo'
MAPPING = {
    'SPEAKER_00': 'Entrevistador',
    'SPEAKER_01': 'Invitada',
}
RUN_RENAME = False  # ⇦ True cuando MAPPING esté listo

if RUN_RENAME:
    stats = rename_speakers_in_outputs(paths, BASE_NAME, MAPPING)
    print(f'\n✅ Renombrado en {len(stats)} archivo(s):')
    for nombre, cuenta in stats.items():
        print(f'   {nombre}: {cuenta} reemplazo(s)')
    entregables = Path(WORKSPACE) / 'entregables'
    if entregables.exists():
        candidatos = []
        if COPY_TRANSCRIPT_MD:
            candidatos.append(paths.transcripts / f'{BASE_NAME}.transcript.md')
        if COPY_FULL_FOR_LLM:
            candidatos.append(paths.splits / f'{BASE_NAME}_full_for_llm.txt')
        refrescados = [s.name for s in candidatos if s.exists()
                       and shutil.copy2(s, entregables / s.name)]
        print(f"   📦 Entregables refrescados: {len(refrescados)}"
              ' (re-ejecuta la CELDA 8 si copias más formatos)')
else:
    print('\n⏩ Edita MAPPING y pon RUN_RENAME=True para aplicar.')

## 🟡 10 · Benchmark contra referencia humana (opcional)

Si transcribiste un archivo a mano (golden), mide WER/DER reales. Requiere `pip install 'speakerscribe[bench]'`. Los resultados quedan en el ledger y alimentan `BENCHMARKS.md`.

In [ ]:
# 🟡 CELDA 10 — bench (opcional): WER/DER contra tu referencia humana
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'jiwer', 'pyannote.metrics'], check=True)
# from speakerscribe.evaluate import bench_run
# bench_run(paths, base_name=BASE_NAME, reference_txt=Path(WORKSPACE) / 'golden' / 'ref.txt')

## 🟢 11 · Liberar memoria GPU (sin cerrar la sesión)

Útil si vas a seguir trabajando (por ejemplo, re-ejecutar la CELDA 8 o renombrar hablantes). Si ya terminaste del todo, sigue con la CELDA 12.

In [9]:
# 🟢 CELDA 11 — limpieza de VRAM (process_batch ya libera; esto es por si quedó algo)
import gc

import torch

from speakerscribe import release_whisper_model

release_whisper_model()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f'💾 VRAM libre: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB '
          f'| en uso: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print('🧹 Memoria liberada. La sesión sigue activa — corre la CELDA 12 para apagar la máquina.')

2026-06-13 00:55:21.119 | DEBUG    | speakerscribe.transcription:release_whisper_model:120 - Whisper reference dropped, CUDA cache cleared


💾 VRAM libre: 12.13 GB | en uso: 0.04 GB
🧹 Memoria liberada. La sesión sigue activa — corre la CELDA 12 para apagar la máquina.


## 🟢 12 · Apagar el runtime (libera la máquina y la cuota de GPU)

Esta celda **termina la sesión de Colab**: sincroniza Drive (para que ninguna salida quede a medio escribir), limpia variables y ejecuta `runtime.unassign()`. Tus archivos en Drive **no** se ven afectados.

Con `APAGAR_AHORA = True` (valor por defecto) la máquina se apaga al ejecutar la celda — ponlo en `False` si todavía no quieres cerrar.

In [10]:
# 🟢 CELDA 12 — apagar el runtime de Colab (libera la máquina)
APAGAR_AHORA = True  # ⇦ ponlo en False si aún NO quieres terminar la sesión

import gc
from datetime import datetime
from zoneinfo import ZoneInfo

print(f"🕐 Hora Bogotá: {datetime.now(ZoneInfo('America/Bogota')):%Y-%m-%d %H:%M:%S}")

# Limpieza final de variables grandes
for _v in list(globals().keys()):
    if _v.startswith(('model', 'results', 'smoke')):
        globals().pop(_v, None)
gc.collect()

if APAGAR_AHORA:
    from google.colab import drive, runtime
    print('💾 Sincronizando Drive (flush_and_unmount)...')
    drive.flush_and_unmount()
    print('🛑 Apagando el runtime — la máquina queda liberada. ¡Hasta la próxima!')
    runtime.unassign()
else:
    print('La sesión sigue activa. Cambia APAGAR_AHORA = True y re-ejecuta para apagar.')

🕐 Hora Bogotá: 2026-06-12 19:55:21
💾 Sincronizando Drive (flush_and_unmount)...
🛑 Apagando el runtime — la máquina queda liberada. ¡Hasta la próxima!


---
## 📖 Referencia rápida

### Escenarios comunes

| Escenario | Solución |
|---|---|
| La CELDA 2 reinició el runtime | Es el reinicio **planificado**: re-ejecuta CELDAS 1–2 (fast-path, segundos) y continúa. |
| Cambiaste el modelo Whisper | El `base_name` incluye el modelo → todo se reprocesa con el nuevo; los outputs del anterior quedan para comparar. |
| La sesión expiró a mitad de batch | Re-ejecuta CELDAS 1–6. El ledger omite automáticamente lo ya completado. |
| Cambiaste `num_speakers` | El caché de diarización se invalida solo (hash de parámetros). |
| Agregaste un audio nuevo | Solo se procesa el nuevo; el resto se omite. |
| Quieres reprocesar todo | `config = config.model_copy(update={'force_reprocess': True})` antes de la CELDA 6. |
| Falló el smoke | Verifica runtime GPU, que el audio no esté corrupto y el token HF. |
| El batch corre pero da `error` en todo | Ejecuta la CELDA 2b: el "origen del código" debe ser site-packages, no Drive. |

### Archivos producidos por audio

| Carpeta | Archivo | Contenido |
|---|---|---|
| `transcripts/` | `<base>.txt` | Texto plano con etiquetas `[SPEAKER_XX]` |
| `transcripts/` | `<base>.srt` | Subtítulos con timestamps |
| `transcripts/` | `<base>.json` | Metadata completa + segmentos (auditoría) |
| `transcripts/` | `<base>.transcript.md` | **Markdown legible para humanos** ⭐ |
| `splits/` | `<base>_N.txt` | Trozos del tamaño de ventana para LLMs de contexto corto |
| `splits/` | `<base>_full_for_llm.txt` | Transcripción completa para LLMs de contexto largo ⭐ |
| `entregables/` | *(selección de la CELDA 3)* | Tu carpeta limpia de consumo + `_resumen.md` + `_metadata/` |

### Resumen de reunión con un LLM

1. Asegúrate de `COPY_FULL_FOR_LLM = True` en la CELDA 3.
2. Abre el `*_full_for_llm.txt` de `entregables/`.
3. Pégalo en Claude con un prompt tipo: *"Con base en la siguiente transcripción de reunión, genera un resumen estructurado: asistentes, decisiones clave, compromisos con responsables y fechas. Transcripción: [pegar]"*.

### Recursos

- 📁 [Repositorio en GitHub](https://github.com/EnriqueForero/speakerscribe) · 🤗 [faster-whisper](https://github.com/SYSTRAN/faster-whisper) · 🤗 [pyannote.audio](https://github.com/pyannote/pyannote-audio)